# Notebook 01

## Phase 1 — Load, Explore & Clean

**Python Foundations + Data Cleaning**

Your first job is to load the data, understand what each column means, and clean up the mess. By the end of this phase you should have a clean DataFrame with no obvious problems.

---

## Tasks

> Create notebook: `01_cleaning.ipynb`

- [ ] Load the data using `pd.read_csv()` and print the first 5 rows
- [ ] Check the shape: how many rows and columns?
- [ ] Inspect data types: use `.info()` — are any columns the wrong type? Fix at least 2
- [ ] Find missing values: use `.isnull().sum()` — which columns have the most? Decide what to do (fill in or drop) and explain why
- [ ] Handle duplicates: check with `.duplicated().sum()` and remove if any
- [ ] Spot outliers: use a boxplot or the IQR method on the target column; cap extreme values at the 99th percentile
- [ ] Write a `clean_data()` function that does all the above steps in order so you can reuse it
- [ ] Add 3 checks at the end (e.g., no nulls in key columns, all target values > 0, correct number of columns)

---

**Tip:** Add a comment or markdown cell explaining every decision. Future-you will thank you!

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv("data/raw/AmesHousing.csv") # read dataset
df.head() # display first 5 rows
# • Load the data using pd.read_csv() and print the first 5 rows

In [ ]:
df.shape # (rows, columns)
# • Check the shape: how many rows and columns?

In [ ]:
cols = ["Bsmt Full Bath", "Bsmt Half Bath"] # columns to be updated

# display info before changes
print("Before changes:")
df[cols].info()

# fill nulls and convert to int
df["Bsmt Full Bath"] = df["Bsmt Full Bath"].fillna(0).astype("int64")
df["Bsmt Half Bath"] = df["Bsmt Half Bath"].fillna(0).astype("int64")

# display info after changes
print("\n\nAfter changes:")
df[cols].info()
# • Inspect data types: use .info() — are any columns the wrong type? Fix at least 2

In [ ]:
print(df.isnull().sum()[df.isnull().sum() > 0]) # display missing values
print("Missing values before:", df.isnull().sum().sum()) # display missing values before changes

# Lot Frontage: Fill with median (safer for outliers)
df["Lot Frontage"] = df["Lot Frontage"].fillna(df["Lot Frontage"].median())

# Electrical: Fill with mode (most frequent value for categorical)
df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

# get columns that still have missing values
missing_cols = [col for col in df.columns if df[col].isnull().any()]

# fill remaining missing: 'None' for categorical, 0 for numeric
for col in missing_cols:
    if df[col].dtype == "object":
        df[col] = df[col].fillna("None") # categorical missing = None
    else:
        df[col] = df[col].fillna(0) # numeric missing = 0
print("Missing values before:", df.isnull().sum().sum()) # display missing values after changes

In [ ]:
print(df.duplicated().sum()) # count duplicates
df = df.drop_duplicates() # remove duplicates
# no duplicates (0)
# • Handle duplicates: check with .duplicated().sum() and remove if any

In [ ]:
q1 = df["SalePrice"].quantile(0.25) # calculate first quartile
q3 = df["SalePrice"].quantile(0.75) # calculate third quartile
iqr = q3 - q1 # iqr formula

lower_limit = q1 - 1.5 * iqr # lower limit
upper_limit = q3 + 1.5 * iqr # upper limit

outliers = ((df["SalePrice"] < lower_limit) | (df["SalePrice"] > upper_limit)).sum() # count outliers
print(f"outliers: {outliers}") # display total

p99 = df["SalePrice"].quantile(0.99) # get the 99th
df["SalePrice"] = df["SalePrice"].clip(upper=p99) # cap at 99th
# • Spot outliers: use a boxplot or the IQR method on the target column; cap extreme values at the 99th percentile

In [ ]:
def clean_data(data: pd.DataFrame) -> pd.DataFrame:
    cleaned = data.copy()
    print(cleaned.head())
    print(cleaned.shape)
    cleaned["Bsmt Full Bath"] = cleaned["Bsmt Full Bath"].fillna(0).astype("int64")
    cleaned["Bsmt Half Bath"] = cleaned["Bsmt Half Bath"].fillna(0).astype("int64")
    cleaned["Lot Frontage"] = cleaned["Lot Frontage"].fillna(cleaned["Lot Frontage"].median())
    cleaned["Electrical"] = cleaned["Electrical"].fillna(cleaned["Electrical"].mode()[0])
    missing_cols = [col for col in cleaned.columns if cleaned[col].isnull().any()]
    for col in missing_cols:
        if cleaned[col].dtype == "object":
            cleaned[col] = cleaned[col].fillna("None")
        else:
            cleaned[col] = cleaned[col].fillna(0)
    print(f"duplicated: {cleaned.duplicated().sum()}")
    cleaned = cleaned.drop_duplicates()
    q1 = cleaned["SalePrice"].quantile(0.25)
    q3 = cleaned["SalePrice"].quantile(0.75)
    iqr = q3 - q1
    lower_limit = q1 - 1.5 * iqr
    upper_limit = q3 + 1.5 * iqr
    outliers = ((cleaned["SalePrice"] < lower_limit) | (cleaned["SalePrice"] > upper_limit)).sum()
    print(f"outliers: {outliers}")
    q99 = cleaned["SalePrice"].quantile(0.99)
    cleaned["SalePrice"] = cleaned["SalePrice"].clip(upper=q99)
    cleaned.info()
    cleaned.to_csv("data/cleaned/cleaned.csv", encoding='cp1252', index=False)
    return cleaned
# • Write a clean_data() function that does all the above steps in order so you can reuse it

In [ ]:
df = clean_data(df)

In [ ]:
def check_data(data: pd.DataFrame) -> pd.DataFrame:
    assert data[["SalePrice", "Overall Qual", "Gr Liv Area"]].isnull().sum().sum() == 0
    assert (data["SalePrice"] > 0).all()
    assert data.shape[1] == 82
# • Add 3 checks at the end (e.g., no nulls in key columns, all target values > 0, correct number of columns)

In [ ]:
check_data(df)